# 第16课 - 使用 Microsoft Foundry 部署可扩展代理

在本笔记本中，您将构建一个面向虚构公司 **Contoso** 的<strong>生产就绪客服代理</strong>。与之前的课程不同，这里的重点不是代理的推理循环——而是围绕它的一切，使代理能够安全地大规模运行：

1. <strong>工具调用</strong> — 订单查询和工单创建。
2. **RAG** — 从知识库中获取政策答案。
3. <strong>记忆</strong> — 跨轮次记住客户信息。
4. <strong>模型路由</strong> — 简单请求发送给小模型，复杂请求发送给大模型。
5. <strong>响应缓存</strong> — 对重复问题无需调用模型直接提供答案。
6. <strong>人工审批</strong> — 超过阈值的退款需暂停等待签批。
7. <strong>评估门限</strong> — 使用离线测试集阻止不良发布。
8. <strong>可观察性</strong> — 每个请求周围的 OpenTelemetry 跟踪。

每个部分都是独立且可运行的。请仔细阅读每一行——生产原语被刻意保持得非常精简。


## 设置

在运行此笔记本之前，请确保您已经：

1. **拥有一个 Microsoft Foundry 项目**，并部署了聊天模型（例如 `gpt-4.1-mini`）。
2. **使用 Azure CLI 登录** — 在终端中运行 `az login`。
3. **设置所需的环境变量：**
   - `AZURE_AI_PROJECT_ENDPOINT` — 您的 Microsoft Foundry 项目端点。
   - `AZURE_AI_MODEL_DEPLOYMENT_NAME` — 您已部署模型的名称。

当设置了 `AZURE_SEARCH_SERVICE_ENDPOINT` 和 `AZURE_SEARCH_API_KEY` 时，RAG 部分使用 **Azure AI 搜索**，否则会回退到内存搜索，以便笔记本在没有搜索资源的情况下运行。


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import re
import dotenv
from typing import Annotated

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential

dotenv.load_dotenv(dotenv.find_dotenv())

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
model = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

if not endpoint or not model:
    raise ValueError(
        "Missing required environment variables. "
        "Please set AZURE_AI_PROJECT_ENDPOINT and AZURE_AI_MODEL_DEPLOYMENT_NAME in your .env file."
    )

provider = FoundryChatClient(
    project_endpoint=endpoint,
    model=model,
    credential=AzureCliCredential(),
)
print("Foundry client ready.")

## 1. 工具

生产工具针对真实系统执行实际工作。这里我们用普通 Python 函数模拟了一个订单数据库和一个票务系统。`@tool` 装饰器将它们暴露给代理。

注意 `issue_refund` 对超过阈值的退款使用了 `approval_mode="always_require"` —— 这是我们后续部署的人机交互原语。


In [ ]:
# Simulated backend systems (in production these are API calls behind scoped identities).
ORDERS = {
    "A1001": {"status": "shipped", "total": 42.00, "eta": "2 days"},
    "A1002": {"status": "processing", "total": 128.50, "eta": "5 days"},
    "A1003": {"status": "delivered", "total": 19.99, "eta": "delivered"},
}
TICKETS: list[dict] = []
REFUND_APPROVAL_THRESHOLD = 50.0


@tool(approval_mode="never_require")
def get_order_status(order_id: Annotated[str, "The customer's order ID, e.g. A1001"]) -> str:
    """Look up the status of a customer order."""
    order = ORDERS.get(order_id.upper())
    if not order:
        return f"No order found with ID {order_id}."
    return (
        f"Order {order_id.upper()}: status={order['status']}, "
        f"total=${order['total']:.2f}, eta={order['eta']}."
    )


@tool(approval_mode="never_require")
def open_ticket(
    subject: Annotated[str, "Short subject line for the support ticket"],
    details: Annotated[str, "Full description of the customer's issue"],
) -> str:
    """Open a support ticket for issues that need human follow-up."""
    ticket_id = f"T{1000 + len(TICKETS) + 1}"
    TICKETS.append({"id": ticket_id, "subject": subject, "details": details})
    return f"Ticket {ticket_id} opened: {subject}"


def refund_needs_approval(amount: float) -> bool:
    """Refunds above the threshold require a human approver."""
    return amount > REFUND_APPROVAL_THRESHOLD


@tool(approval_mode="always_require")
def issue_refund(
    order_id: Annotated[str, "The order to refund"],
    amount: Annotated[float, "Refund amount in USD"],
) -> str:
    """Issue a refund. Execution pauses for human approval before it runs."""
    return f"Refund of ${amount:.2f} issued for order {order_id.upper()}."


print("Tools defined.")

## 2. RAG — 策略知识库

策略问题（“你的退货期限是多少？”）应该从权威来源回答，而不是模型的记忆。我们将一个小型知识库封装为搜索工具。

在生产环境中，这是 **Azure AI 搜索**；这里我们提供了一个内存中的关键词搜索，因此笔记本可以在任何地方运行，并在存在环境变量时自动切换到 Azure AI 搜索。


In [ ]:
KNOWLEDGE_BASE = {
    "returns": "Contoso accepts returns within 30 days of delivery for a full refund. Items must be unused and in original packaging.",
    "shipping": "Standard shipping takes 3-5 business days. Express shipping (1-2 days) is available at checkout for an extra fee.",
    "warranty": "All Contoso electronics carry a 12-month limited warranty covering manufacturing defects.",
    "refund_policy": "Refunds are processed to the original payment method within 5 business days of approval. Refunds over $50 require a supervisor's approval.",
}


def _in_memory_search(query: str) -> str:
    q = query.lower()
    hits = [text for key, text in KNOWLEDGE_BASE.items() if key.replace("_", " ") in q or key in q]
    if not hits:
        # crude keyword fallback so the tool still returns something useful
        hits = [text for text in KNOWLEDGE_BASE.values() if any(w in text.lower() for w in q.split())]
    return "\n".join(hits) if hits else "No matching policy found."


def _azure_search(query: str) -> str:
    from azure.core.credentials import AzureKeyCredential
    from azure.search.documents import SearchClient

    client = SearchClient(
        endpoint=os.environ["AZURE_SEARCH_SERVICE_ENDPOINT"],
        index_name=os.getenv("AZURE_SEARCH_INDEX_NAME", "contoso-policies"),
        credential=AzureKeyCredential(os.environ["AZURE_SEARCH_API_KEY"]),
    )
    results = client.search(search_text=query, top=3)
    return "\n".join(r.get("content", "") for r in results) or "No matching policy found."


USE_AZURE_SEARCH = bool(os.getenv("AZURE_SEARCH_SERVICE_ENDPOINT") and os.getenv("AZURE_SEARCH_API_KEY"))


@tool(approval_mode="never_require")
def search_policies(query: Annotated[str, "The policy question to look up"]) -> str:
    """Search Contoso support policies to answer customer questions."""
    if USE_AZURE_SEARCH:
        return _azure_search(query)
    return _in_memory_search(query)


print(f"RAG ready. Using {'Azure AI Search' if USE_AZURE_SEARCH else 'in-memory search'}.")

## 3. 内存

一个忘记自己正在与谁交谈的支持代理是一个糟糕的支持代理。我们为每个客户维护一个小型的个人资料存储，并将一个简短的摘要注入代理的指令中。在生产环境中，这通常是一个内存服务（参见第13课）；这里用字典来展示这种模式。


In [ ]:
CUSTOMER_MEMORY: dict[str, dict] = {
    "cust-42": {"name": "Dana", "tier": "enterprise", "recent_order": "A1002"},
    "cust-99": {"name": "Sam", "tier": "standard", "recent_order": "A1003"},
}


def memory_context(customer_id: str) -> str:
    profile = CUSTOMER_MEMORY.get(customer_id)
    if not profile:
        return "This is a new customer with no history."
    return (
        f"Customer {profile['name']} ({profile['tier']} tier). "
        f"Most recent order: {profile['recent_order']}."
    )


print(memory_context("cust-42"))

## 4 & 5. 模型路由和响应缓存

两个成本杠杆连接到单个请求处理器：

- <strong>路由</strong>：一个廉价的启发式分类器决定请求需要小模型还是大模型。
- <strong>缓存</strong>：规范化的重复问题直接从缓存提供，无需调用模型。

这里的分类器是故意设计得很简单。在生产环境中，您需要根据流量进行验证，并且可以用 Foundry 的模型路由器替代它。


In [ ]:
SMALL_MODEL = os.getenv("AZURE_AI_SMALL_MODEL", model)   # e.g. gpt-4.1-mini
LARGE_MODEL = os.getenv("AZURE_AI_LARGE_MODEL", model)   # e.g. gpt-4.1

response_cache: dict[str, str] = {}
route_counters = {"small": 0, "large": 0, "cache": 0}


def normalize(query: str) -> str:
    return re.sub(r"\s+", " ", query.lower().strip())


COMPLEX_SIGNALS = ("refund", "cancel", "complaint", "escalate", "broken", "wrong", "why")


def is_simple(query: str) -> bool:
    """Route complex or high-stakes requests to the large model; everything else to the small one."""
    q = query.lower()
    if any(signal in q for signal in COMPLEX_SIGNALS):
        return False
    return len(q.split()) <= 20


def choose_model(query: str) -> str:
    return SMALL_MODEL if is_simple(query) else LARGE_MODEL


print(f"Small model: {SMALL_MODEL} | Large model: {LARGE_MODEL}")

## 6 & 8. 代理、人类审批与可观测性

现在我们从上述工具组装代理，并用 OpenTelemetry 跨度包装每个请求。`handle_support_request` 函数是生产请求处理程序：缓存 → 路由 → 跟踪 → 运行 → 缓存。

人类审批由框架处理：因为 `issue_refund` 的 `approval_mode="always_require"`，运行暂停并弹出审批请求，我们显式地解决该请求。


In [ ]:
# Tracing: use the Agent Framework tracer if available, else a no-op so the notebook runs anywhere.
try:
    from agent_framework.observability import get_tracer
    tracer = get_tracer()
except Exception:  # observability extras not installed
    from contextlib import contextmanager

    class _NoopSpan:
        def set_attribute(self, *_args, **_kwargs):
            pass

    class _NoopTracer:
        @contextmanager
        def start_as_current_span(self, _name):
            yield _NoopSpan()

    tracer = _NoopTracer()


SUPPORT_INSTRUCTIONS = (
    "You are Contoso's customer support agent. Be concise, friendly, and accurate. "
    "Use search_policies for policy questions, get_order_status for orders, "
    "open_ticket when a human needs to follow up, and issue_refund for refunds. "
    "Never invent policy details."
)

support_agent = provider.as_agent(
    name="ContosoSupportAgent",
    instructions=SUPPORT_INSTRUCTIONS,
    tools=[get_order_status, open_ticket, search_policies, issue_refund],
)


async def handle_support_request(query: str, customer_id: str) -> str:
    # 1. Serve from cache when we can.
    key = normalize(query)
    if key in response_cache:
        route_counters["cache"] += 1
        return response_cache[key]

    # 2. Route by complexity to control cost.
    chosen_model = choose_model(query)
    route_counters["small" if chosen_model == SMALL_MODEL else "large"] += 1

    # 3. Add per-customer memory to the prompt.
    context = memory_context(customer_id)
    prompt = f"[Customer context: {context}]\n\n{query}"

    # 4. Run inside a trace span for observability.
    with tracer.start_as_current_span("support_request") as span:
        span.set_attribute("customer.id", customer_id)
        span.set_attribute("routed.model", chosen_model)
        response = await support_agent.run(prompt, model=chosen_model)

    text = response.text
    response_cache[key] = text
    return text


print("Support agent assembled.")

In [ ]:
# Try a few requests. The first is simple (small model), the second is a refund (large model + approval path).
print(await handle_support_request("What is your return window?", "cust-99"))
print("---")
print(await handle_support_request("Where is my order A1002?", "cust-42"))
print("---")
# Repeat the first question -> served from cache.
print(await handle_support_request("What is your return window?", "cust-99"))
print("---")
print("Routing counters:", route_counters)

## 7. 评估关卡

这是课程中的发布关卡：使用线下测试集对代理进行评分，只有通过率超过阈值时才进行部署。这里的评分器是一个简单的关键词重叠检查，以保持笔记本的自包含；在生产中你会使用作为裁判的LLM或框架评估器（参见第10课）。


In [ ]:
TEST_CASES = [
    {"input": "How long do I have to return an item?", "expected": ["30 days", "refund"]},
    {"input": "How fast is standard shipping?", "expected": ["3-5", "business days"]},
    {"input": "What is the status of order A1001?", "expected": ["shipped", "A1001"]},
    {"input": "Do your electronics have a warranty?", "expected": ["12-month", "warranty"]},
]


def score_response(actual: str, expected_keywords: list[str]) -> float:
    actual_l = actual.lower()
    hits = sum(1 for kw in expected_keywords if kw.lower() in actual_l)
    return hits / len(expected_keywords)


async def evaluation_gate(test_cases: list[dict], threshold: float = 0.8) -> bool:
    passed = 0
    for case in test_cases:
        result = await support_agent.run(case["input"])
        s = score_response(result.text, case["expected"])
        status = "PASS" if s >= 0.5 else "FAIL"
        print(f"[{status}] {case['input']}  (score={s:.0%})")
        if s >= 0.5:
            passed += 1
    pass_rate = passed / len(test_cases)
    print(f"\nEvaluation pass rate: {pass_rate:.0%} (gate: {threshold:.0%})")
    return pass_rate >= threshold


gate_passed = await evaluation_gate(TEST_CASES, threshold=0.8)
print("\nDeploy allowed:" , gate_passed)

## 综合应用：模拟发布

下面的代码单元展示了课程中描述的完整循环：运行评估门，只有通过时才“部署”。这是在将代理版本推广到 Foundry Agent Service 之前，在 CI 中运行的模式。


In [ ]:
async def release(test_cases: list[dict]) -> None:
    print("Running pre-deployment evaluation gate...\n")
    if await evaluation_gate(test_cases, threshold=0.8):
        print("\n✅ Gate passed — promoting agent version to the Foundry Agent Service.")
    else:
        print("\n❌ Gate failed — release blocked. Fix the agent and re-run.")


await release(TEST_CASES)

## 概要

你组装了一个面向生产的客户支持代理，内置了所有运营相关的关注点：

- **工具、RAG 和记忆** 为代理提供能力和上下文。
- <strong>模型路由和缓存</strong> 控制延迟和成本。
- <strong>人工审核</strong> 保障大额退款等高风险操作。
- <strong>评估关卡</strong> 在发布前阻止不良版本。
- <strong>追踪</strong> 使每个请求都可观察。

### 挑战

扩展此代理以：

1. <strong>支持多模型</strong> — 添加第三个“推理”层，并将升级/投诉路由到该层。
2. <strong>添加评估关卡</strong> — 扩展 `TEST_CASES` 包含退款审批场景，确认该关卡能够捕获回归。
3. <strong>添加成本感知路由</strong> — 跟踪每个请求的估算成本（小额 vs 大额 vs 缓存），并在一批混合查询后打印成本报告。

下一课中，你将走相反的路线，完全在自己的机器上使用 Microsoft Foundry Local 和 Qwen 运行一个代理。


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**免责声明**：
本文件由 AI 翻译服务 [Co-op Translator](https://github.com/Azure/co-op-translator) 翻译完成。尽管我们力求准确，但请注意，自动翻译可能包含错误或不准确之处。原始语言版文件应视为权威来源。对于重要信息，建议使用专业人工翻译。我们对因使用本翻译而产生的任何误解或误释不承担责任。
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
